# Gold | Demonstração de falha de DQ (evidência para o vídeo / entrega)

O Tech Challenge pede **evidência de falha detectada**. Este notebook cria uma cópia **descartável** de uma fato em
`workspace.gold_staging_demo`, injeta duplicidade e município inexistente, roda as regras com `layer = GOLD_DEMO`
e **espera** que o gate bloqueie. Não toca em `gold_staging` nem em `gold`; os registros `GOLD_DEMO` ficam em
`monitoring.qualidade_resultados` para o alerta/dashboard mostrarem.
Resultado esperado: o notebook termina com sucesso **somente se** o gate bloqueou.

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold_staging_demo")
spark.sql("""CREATE OR REPLACE TABLE workspace.gold_staging_demo.ft AS
             SELECT * FROM workspace.gold_staging.ft_resultado_municipio LIMIT 1000""")
spark.sql("""INSERT INTO workspace.gold_staging_demo.ft
             SELECT * FROM workspace.gold_staging_demo.ft LIMIT 5""")               # duplicidade de chave
spark.sql("""INSERT INTO workspace.gold_staging_demo.ft
             SELECT ano, '9999999' AS id_municipio, serie, rede_codigo, taxa_alfabetizacao, media_portugues,
                    proporcao_aluno_nivel_0, proporcao_aluno_nivel_1, proporcao_aluno_nivel_2,
                    proporcao_aluno_nivel_3, proporcao_aluno_nivel_4, proporcao_aluno_nivel_5,
                    proporcao_aluno_nivel_6, proporcao_aluno_nivel_7, proporcao_aluno_nivel_8
             FROM workspace.gold_staging_demo.ft LIMIT 200""")                       # município órfão

In [0]:
_m = iniciar_etapa()
dq = RegistroDQ("GOLD_DEMO", limiar_correspondencia=95)
dq.chave_nula_e_duplicada("workspace.gold_staging_demo.ft", ["ano", "id_municipio", "serie", "rede_codigo"])
dq.taxa_correspondencia("demo ft x dim_municipio", "workspace.gold_staging_demo.ft",
                        "workspace.gold_staging.dim_municipio", "l.id_municipio = r.id_municipio")

bloqueou = False
try:
    dq.gate()
except RuntimeError as exc:
    bloqueou = True
    registrar_execucao("GOLD_DEMO", "dq_gold_demo", _m, status="FALHA", erro=exc)
    print("Gate bloqueou como esperado:", exc)

spark.sql("DROP SCHEMA IF EXISTS workspace.gold_staging_demo CASCADE")
assert bloqueou, "O teste de falha NÃO bloqueou: as regras de DQ não estão detectando o defeito injetado."